# Pixel Art LoRA: train, benchmark, publish

Run this on **Kaggle** with:
- **Accelerator:** GPU T4 x2 (Settings panel on the right)
- **Internet:** On

Steps: install → build the training set → train the LoRA → benchmark → upload the weights.

## 1. Get the code and install libraries

In [ ]:
!git clone https://github.com/ABDULAH-ab/Pixel-Art.git
%cd Pixel-Art
!pip install -q -U diffusers peft accelerate transformers datasets bitsandbytes
# Kaggle ships an old torchao that the new peft refuses to load. We do not use it, so remove it.
!pip uninstall -y -q torchao

## 2. Build the training set

Streams the Pixel Art Atlas dataset and keeps 1000 sprites, one per subject.
Each 16x16 sprite is pasted on white and upscaled to 1024x1024 with nearest-neighbour,
so every sprite pixel becomes a sharp 64x64 block.

In [ ]:
!python scripts/prepare_data.py --n 1000

In [ ]:
# Look at a few training images and their captions before spending GPU time on them.
import json
from PIL import Image
from IPython.display import display

rows = [json.loads(line) for line in open("data/train/metadata.jsonl")][:6]
strip = Image.new("RGB", (256 * len(rows), 256), "white")
for i, row in enumerate(rows):
    strip.paste(Image.open(f"data/train/{row['file_name']}").resize((256, 256), Image.Resampling.NEAREST), (i * 256, 0))
display(strip)
for row in rows:
    print(row["text"])

## 3. Train the LoRA

We use the official `diffusers` training script rather than writing our own loop.
It is downloaded from the tag that matches the installed `diffusers` version, so the two always agree.

What the main flags mean:
- `--rank=32`: size of the LoRA matrices. Higher = more capacity, bigger file.
- `--max_train_steps=3000`: with batch size 1, that is 3 passes over 1000 images.
- `--snr_gamma=5.0`: Min-SNR loss weighting. Stops the easy, very noisy steps from dominating the loss, so training converges faster.
- `--noise_offset=0.05`: lets the model learn very bright or very dark images. Without it SDXL drifts towards mid-grey backgrounds instead of white.
- `--mixed_precision=fp16`: half precision, roughly halves memory use.
- `--gradient_checkpointing` and `--use_8bit_adam`: trade a little speed for less memory so SDXL fits on a 16 GB T4.

If you run out of memory, change `--resolution` to 768.

In [ ]:
import urllib.request
import diffusers

url = f"https://raw.githubusercontent.com/huggingface/diffusers/v{diffusers.__version__}/examples/text_to_image/train_text_to_image_lora_sdxl.py"
urllib.request.urlretrieve(url, "train_text_to_image_lora_sdxl.py")
print("diffusers", diffusers.__version__)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 accelerate launch train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0 \
  --pretrained_vae_model_name_or_path=madebyollin/sdxl-vae-fp16-fix \
  --train_data_dir=data/train \
  --caption_column=text \
  --resolution=1024 \
  --image_interpolation_mode=nearest \
  --train_batch_size=1 \
  --max_train_steps=3000 \
  --learning_rate=1e-4 \
  --lr_scheduler=constant \
  --lr_warmup_steps=0 \
  --rank=32 \
  --mixed_precision=fp16 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --checkpointing_steps=1000 \
  --snr_gamma=5.0 \
  --noise_offset=0.05 \
  --seed=0 \
  --output_dir=outputs/pixel-lora

## 4. Benchmark

Runs four configurations on the same 8 prompts and seeds, and measures latency,
peak VRAM, CLIP score, grid error and palette size.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/benchmark.py --lora outputs/pixel-lora

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
from PIL import Image

display(Markdown(Path("outputs/benchmark/results.md").read_text()))

# README image. Right-click it and choose "Save image as" to get the original file.
display(Image.open("outputs/benchmark/comparison.png"))

for path in sorted(Path("outputs/benchmark").glob("grid_*.png")):
    print(path.stem)
    display(Image.open(path))

### Sprite showcase

Turns the base model's and the LoRA's outputs into real 16x16 sprites (snap to grid, remove background) and places them in a game-style inventory bar. No GPU needed.

In [ ]:
!python scripts/make_showcase.py
display(Image.open("outputs/benchmark/showcase.png"))

## 5. Upload the LoRA to the Hugging Face Hub

First add your Hugging Face **write** token in Kaggle: *Add-ons → Secrets → Add secret*, name it `HF_TOKEN`.

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
repo_id = f"{api.whoami()['name']}/pixel-art-sdxl-lora"
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(repo_id=repo_id, folder_path="outputs/pixel-lora", allow_patterns=["*.safetensors"])
print("Uploaded to https://huggingface.co/" + repo_id)

## 6. Download the benchmark results

The zip appears in the Output panel on the right. The images and table go into the README.

In [ ]:
!zip -q -r benchmark_results.zip outputs/benchmark && ls -lh benchmark_results.zip